### BLIBIOTECA

In [16]:
import pandas as pd
from extractor import extract_all_files, inspect, extract_summary, check_duplicates
from transform import cast_types, drop_nulls, fill_nulls, add_columns, filter_rows, joins
from storage import save_csv, save_parquet

### CAMINHOS E CONFIGURAÇÕES

In [17]:
RAW_PATH       = r"C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Raw"
PROCESSED_PATH = r"C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed"

In [18]:
FILES = {
    "deliveries": {
        "file":   "deliveries.csv",
        "schema": ["delivery_id", "driver_id", "hub_id", "status",
                   "shipped_date", "delivered_date", "distance_km",
                   "cost", "customer_rating"],
    },
    "drivers": {
        "file":   "drivers.csv",
        "schema": ["driver_id", "name", "hub_id"],
    },
    "hubs": {
        "file":   "hubs.csv",
        "schema": ["hub_id", "city", "state"],
    },
}

### EXTRACT

In [19]:
print("\n===EXTRACT===")
dfs = extract_all_files(RAW_PATH, FILES)


===EXTRACT===
[extract] deliveries.csv | 102000 linhas | 11 colunas
[extract] drivers.csv | 255 linhas | 3 colunas
[extract] hubs.csv | 10 linhas | 3 colunas


In [20]:
print("\n=== VALIDAR ===")
check_duplicates(dfs["deliveries"], "delivery_id", "deliveries")
check_duplicates(dfs["drivers"],    "driver_id",   "drivers")
check_duplicates(dfs["hubs"],       "hub_id",      "hubs")


=== VALIDAR ===
[validate] 'deliveries' - duplicatas em 'delivery_id': 2000
[validate] 'drivers' - duplicatas em 'driver_id': 0
[validate] 'hubs' - duplicatas em 'hub_id': 0


np.int64(0)

### TRANSFORM DELIVERES

In [21]:
print("\n===TRANSFORM: DELIVERIES===")
df = dfs["deliveries"].copy()


===TRANSFORM: DELIVERIES===


In [22]:
df = cast_types(df, {
    "shipped_date":   "datetime",
    "delivered_date": "datetime",
    "distance_km":    "float",
    "cost":           "float",
    "customer_rating":"float",
})

[transform] 'shipped_date' -> datetime
[transform] 'delivered_date' -> datetime
[transform] 'distance_km' -> float
[transform] 'cost' -> float
[transform] 'customer_rating' -> float


In [23]:
df = filter_rows(df, {"status": ["delivered", "in_transit", "pending"]}, "deliveries")

[transform] 'deliveries' — filter {'status': ['delivered', 'in_transit', 'pending']}: 68049 linhas removidas


In [24]:
df = fill_nulls(df, {
    "customer_rating": 0.0,
    "delivered_date":  pd.NaT,
})

[transform] 'customer_rating' - 1732 nulos preenchidos com '0.0'
[transform] 'delivered_date' - 1672 nulos preenchidos com 'NaT'


In [25]:
df = add_columns(df, {
    "delivery_days": lambda d: (d["delivered_date"] - d["shipped_date"]).dt.days,
    "cost_per_km":   lambda d: (d["cost"] / d["distance_km"]).round(2),
})

[transform] Coluna 'delivery_days' criada
[transform] Coluna 'cost_per_km' criada


### TRANSFORM DRIVERS

In [26]:
print("\n===TRANSFORM: DRIVERS===")
df_drivers = dfs["drivers"].copy()
df_drivers["name"] = df_drivers["name"].str.strip().str.title()


===TRANSFORM: DRIVERS===


### TRANSFORM HUBS

In [27]:
print("\n===TRANSFORM: HUBS===")
df_hubs = dfs["hubs"].copy()
df_hubs["city"]  = df_hubs["city"].str.strip().str.title()
df_hubs["state"] = df_hubs["state"].str.strip().str.upper()


===TRANSFORM: HUBS===


In [28]:
print("\n===JOIN's===")
df = joins(df, df_drivers, on="driver_id", cols=["driver_id", "name"])
df = joins(df, df_hubs,    on="hub_id",    cols=["hub_id", "city", "state"])
df = df.rename(columns={"name": "driver_name"})


===JOIN's===
[transform] enrich on 'driver_id' (left) - shape: (33951, 14)
[transform] enrich on 'hub_id' (left) - shape: (33951, 16)


In [29]:
print("\n=== RESULTADO FINAL ===")
inspect(df, "deliveries_transformed")


=== RESULTADO FINAL ===

[inspect] deliveries_transformed

Amostra:
   delivery_id  driver_id  hub_id     status shipped_date delivered_date  \
0            6        266       1  delivered   2026-04-16     2026-04-17   
1            7        173       5  delivered   2026-02-01     2026-02-06   
2           13        217      10  delivered   2026-01-01     2026-01-02   
3           14        395       8  delivered   2026-04-11     2026-04-15   
4           15        306       6  delivered   2026-02-01     2026-02-03   

   distance_km    cost  customer_rating  created_at  updated_at  \
0        67.12  571.51              1.9  2026-04-11  2026-04-19   
1       429.01  368.38              1.6  2026-02-01  2026-02-02   
2        68.33  337.12              2.1  2025-12-27  2026-01-05   
3        97.83  429.34              3.9  2026-04-11  2026-04-11   
4       422.46  257.31              2.8  2026-02-01  2026-02-06   

   delivery_days  cost_per_km driver_name     city state  
0           

### LOAD

In [30]:
print("\n===SAVE===")
save_parquet(df, f"{PROCESSED_PATH}\\transformed_deliveries.parquet")
save_csv(df,     f"{PROCESSED_PATH}\\transformed_deliveries.csv")


===SAVE===
[storage] Parquet salvo em 'C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed\transformed_deliveries.parquet' — 33951 linhas
[storage] CSV salvo em 'C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed\transformed_deliveries.csv' — 33951 linhas
